![DepthDif banner](https://raw.githubusercontent.com/simon-donike/DepthDif/main/docs/assets/branding/banner_depthdif.png)

# 🌊 DepthDif ocean-temperature explorer

Welcome! This notebook turns scattered ARGO ocean measurements into a smooth temperature map for the place and week you choose. You can optionally add OSTIA sea-surface temperature for extra surface context, or skip it and let the model work from ARGO alone. 🧭

The notebook takes care of the behind-the-scenes pieces for you: it fetches the public DepthDif model, prepares the land mask, downloads the matching ARGO files, and then paints the result back onto an interactive map. If you choose OSTIA, you will need a Copernicus Marine login; if you press **Skip OSTIA**, no Copernicus login is needed. ✨


## ⚙️ 1. Get the notebook ready
This first step installs the tools the notebook needs and prepares the interactive map widgets. It may take a minute, but you only need to run it once per Colab session. ☕

In [ ]:
%pip install -q "numpy<2.1" "click>=8.2.1" depth-recon copernicusmarine ipywidgets ipyleaflet


In [ ]:
# @title 2. Hidden notebook helpers { display-mode: "form" }
from __future__ import annotations

import base64
from datetime import date
from io import BytesIO
import json
import os
from pathlib import Path

from IPython.display import HTML, clear_output, display
from ipyleaflet import DrawControl, GeoJSON, ImageOverlay, LayersControl, Map, basemaps
import ipywidgets as widgets
from matplotlib.colors import LinearSegmentedColormap, Normalize
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.warp import transform_bounds
import yaml

from depth_recon import resolve_hf_assets, run_week_inference

try:
    from google.colab import output

    # Colab needs this for third-party ipywidgets such as ipyleaflet.
    output.enable_custom_widget_manager()
except Exception:
    pass

DEPTH_LEVEL_OPTIONS = (
    (0, 'surface', 'Surface (0 m)'),
    (10, '10m', '10 m'),
    (50, '50m', '50 m'),
    (100, '100m', '100 m'),
    (250, '250m', '250 m'),
    (500, '500m', '500 m'),
    (1000, '1000m', '1000 m'),
    (2000, '2000m', '2000 m'),
    (2500, '2500m', '2500 m'),
)
TEMPERATURE_RANGE_C = (0.0, 30.0)
TEMPERATURE_CMAP = LinearSegmentedColormap.from_list(
    'depthdif_temperature', ['#1d4ed8', '#facc15', '#dc2626']
)

STATE = {
    'hf_repo_id': 'simon-donike/DepthDif',
    'hf_revision': 'main',
    'cache_dir': Path('/content/depthdif_cache'),
    'output_root': Path('/content/depthdif_outputs'),
    'device': 'auto',
    'batch_size': 4,
    'skip_ostia': False,
    'rectangle': None,
    'assets': None,
    'run_dir': None,
}


def _html_status(message: str, kind: str = 'info') -> HTML:
    """Return a compact notebook status message."""
    colors = {'info': '#334155', 'ok': '#166534', 'warn': '#92400e'}
    color = colors.get(kind, colors['info'])
    return HTML(f'<span style="color:{color}">{message}</span>')


def iso_weeks_in_year(year: int) -> list[int]:
    """Return the valid ISO week numbers for one calendar year."""
    return list(range(1, date(int(year), 12, 28).isocalendar().week + 1))


def _runtime_widgets() -> dict[str, widgets.Widget]:
    """Create or return the runtime configuration widgets."""
    if 'runtime_widgets' not in STATE:
        STATE['runtime_widgets'] = {
            'cache_dir': widgets.Text(
                value=str(STATE['cache_dir']), description='Cache', layout=widgets.Layout(width='520px')
            ),
            'output_root': widgets.Text(
                value=str(STATE['output_root']), description='Outputs', layout=widgets.Layout(width='520px')
            ),
            'device': widgets.Dropdown(options=['auto', 'cuda', 'cpu'], value=str(STATE['device']), description='Device'),
            'batch_size': widgets.BoundedIntText(value=int(STATE['batch_size']), min=1, max=128, description='Batch size'),
        }
    return STATE['runtime_widgets']


def _sync_runtime_settings() -> None:
    """Copy current runtime widget values into STATE."""
    runtime = _runtime_widgets()
    STATE['cache_dir'] = Path(runtime['cache_dir'].value).expanduser()
    STATE['output_root'] = Path(runtime['output_root'].value).expanduser()
    STATE['device'] = runtime['device'].value
    STATE['batch_size'] = int(runtime['batch_size'].value)


def configure_depthdif_runtime() -> None:
    """Display runtime options that are safe for most Colab users to change."""
    runtime = _runtime_widgets()
    for widget in runtime.values():
        widget.observe(lambda change: _sync_runtime_settings(), names='value')
    _sync_runtime_settings()
    display(widgets.VBox([
        widgets.HTML(
            f'Hugging Face release: <b>{STATE["hf_repo_id"]}</b> @ <b>{STATE["hf_revision"]}</b>'
        ),
        runtime['cache_dir'],
        runtime['output_root'],
        widgets.HBox([runtime['device'], runtime['batch_size']]),
    ]))


def _auth_widgets() -> dict[str, widgets.Widget]:
    """Create or return Copernicus Marine credential widgets."""
    if 'auth_widgets' not in STATE:
        STATE['auth_widgets'] = {
            'username': widgets.Text(
                value=os.environ.get('COPERNICUSMARINE_SERVICE_USERNAME', ''),
                description='Username',
                layout=widgets.Layout(width='420px'),
            ),
            'token': widgets.Password(
                value=os.environ.get('COPERNICUSMARINE_SERVICE_PASSWORD', ''),
                description='Token/key',
                layout=widgets.Layout(width='420px'),
            ),
            'button': widgets.Button(description='Load credentials', icon='key', button_style='primary'),
            'skip_button': widgets.Button(description='Skip OSTIA', icon='step-forward'),
            'status': widgets.HTML(''),
        }
    return STATE['auth_widgets']


def _sync_copernicus_environment(required: bool = False) -> tuple[str, str]:
    """Store Copernicus credentials in environment variables for the current runtime."""
    auth = _auth_widgets()
    username = auth['username'].value.strip()
    token = auth['token'].value
    if required and (not username or not token):
        raise ValueError('Please enter your Copernicus Marine username and token/key, or press Skip OSTIA to continue with ARGO only.')
    if username:
        os.environ['COPERNICUSMARINE_SERVICE_USERNAME'] = username
    if token:
        # The toolbox reads the API key/token from its password environment variable.
        os.environ['COPERNICUSMARINE_SERVICE_PASSWORD'] = token
    return username, token


def authenticate_copernicusmarine() -> None:
    """Open Copernicus Marine username and token inputs for OSTIA downloads."""
    auth = _auth_widgets()

    def handle_click(_button: widgets.Button) -> None:
        _sync_copernicus_environment(required=True)
        STATE['skip_ostia'] = False
        auth['status'].value = '<span style="color:#166534">✅ Credentials loaded. OSTIA will add sea-surface context to this run.</span>'

    def handle_skip(_button: widgets.Button) -> None:
        STATE['skip_ostia'] = True
        auth['status'].value = '<span style="color:#92400e">⏭️ OSTIA skipped. The run will use ARGO observations only.</span>'

    auth['button'].on_click(handle_click)
    auth['skip_button'].on_click(handle_skip)
    display(widgets.VBox([
        widgets.HTML('🌡️ OSTIA adds a sea-surface temperature layer. Load your Copernicus Marine credentials to include it, or choose Skip OSTIA for a simpler ARGO-only run.'),
        auth['username'],
        auth['token'],
        widgets.HBox([auth['button'], auth['skip_button'], auth['status']]),
    ]))


def rectangle_from_geojson(geo_json: dict) -> tuple[float, float, float, float]:
    """Extract lon_min, lat_min, lon_max, lat_max from a drawn rectangle."""
    geometry = geo_json.get('geometry', geo_json)
    if geometry.get('type') == 'FeatureCollection':
        geometry = geometry['features'][0]['geometry']
    coords = geometry['coordinates'][0]
    lons = [float(point[0]) for point in coords]
    lats = [float(point[1]) for point in coords]
    return min(lons), min(lats), max(lons), max(lats)


def select_week_and_bbox() -> None:
    """Display ISO week controls and a Leaflet rectangle picker."""
    year_options = list(range(2010, date.today().year + 1))
    year_widget = widgets.Dropdown(options=year_options, value=2015, description='Year')
    week_widget = widgets.Dropdown(options=iso_weeks_in_year(2015), value=25, description='ISO week')
    rectangle_status = widgets.HTML('🖊️ Draw one rectangle on the map to choose your ocean area.')
    continue_button = widgets.Button(description='Continue', icon='check', button_style='primary')
    draw_map = Map(
        center=(20.0, 0.0),
        zoom=2,
        basemap=basemaps.Esri.WorldImagery,
        scroll_wheel_zoom=True,
    )
    draw_control = DrawControl(
        rectangle={'shapeOptions': {'color': '#0f766e', 'fillOpacity': 0.12}},
        polygon={},
        polyline={},
        circle={},
        circlemarker={},
        marker={},
    )

    def sync_week_options(change: dict) -> None:
        """Keep the ISO-week dropdown valid when the year changes."""
        weeks = iso_weeks_in_year(int(change['new']))
        current_week = int(week_widget.value)
        week_widget.options = weeks
        week_widget.value = current_week if current_week in weeks else weeks[-1]

    def handle_draw(target: DrawControl, action: str, geo_json: dict) -> None:
        """Store the latest rectangle drawn in the Leaflet widget."""
        if action in {'created', 'edited'}:
            bounds = rectangle_from_geojson(geo_json)
            STATE['rectangle'] = bounds
            rectangle_status.value = (
                '✅ Area selected: '
                f'lon {bounds[0]:.3f} to {bounds[2]:.3f}, '
                f'lat {bounds[1]:.3f} to {bounds[3]:.3f}'
            )
        elif action == 'deleted':
            STATE['rectangle'] = None
            rectangle_status.value = '🖊️ Draw one rectangle on the map to choose your ocean area.'

    def handle_continue(_button: widgets.Button) -> None:
        """Confirm the selected week and rectangle before inference."""
        if STATE.get('rectangle') is None:
            rectangle_status.value = '<span style="color:#92400e">🖊️ Please draw one rectangle on the map before continuing.</span>'
            return
        rectangle_status.value = '<span style="color:#166534">✅ Selection saved. You are ready to run the model.</span>'

    year_widget.observe(sync_week_options, names='value')
    draw_control.on_draw(handle_draw)
    continue_button.on_click(handle_continue)
    draw_map.add_control(draw_control)
    STATE['year_widget'] = year_widget
    STATE['week_widget'] = week_widget
    display(widgets.HBox([year_widget, week_widget, continue_button]), rectangle_status, draw_map)


def resolve_depthdif_assets() -> None:
    """Resolve the Hugging Face config files and checkpoint used for inference."""
    _sync_runtime_settings()
    assets = resolve_hf_assets(
        config_repo=STATE['hf_repo_id'],
        revision=STATE['hf_revision'],
        cache_dir=STATE['cache_dir'],
        force_download=False,
    )
    STATE['assets'] = assets
    display(_html_status('✅ Model files are ready.', 'ok'))
    display(HTML(
        '<br>'.join([
            f'Model config: {assets.model_config}',
            f'Data config: {assets.data_config}',
            f'Train config: {assets.train_config}',
            f'Checkpoint: {assets.checkpoint}',
        ])
    ))


def prediction_tif_from_run(run_dir: Path, suffix: str = 'surface') -> Path:
    """Resolve one exported prediction GeoTIFF from a DepthDif run directory."""
    run_dir = Path(run_dir)
    summary_path = run_dir / 'run_summary.yaml'
    if summary_path.exists():
        summary = yaml.safe_load(summary_path.read_text()) or {}
        for record in summary.get('depth_exports', []):
            label = str(record.get('label', '')).lower()
            record_suffix = str(record.get('suffix', '')).lower()
            if suffix.lower() in {label, record_suffix}:
                path = Path(record['prediction_tif_path'])
                return path if path.is_absolute() else run_dir / path
        if summary.get('prediction_tif_path'):
            path = Path(summary['prediction_tif_path'])
            return path if path.is_absolute() else run_dir / path
    matches = sorted(run_dir.glob(f'*_prediction_{suffix}.tif'))
    if not matches:
        matches = sorted(run_dir.glob('*_prediction*.tif'))
    if not matches:
        raise FileNotFoundError(f'No prediction map was found in {run_dir}. Please run inference first.')
    return matches[0]


def depth_level_metadata(run_dir: Path, depth_m: int) -> tuple[str, str]:
    """Return the exported raster suffix and display label for a requested depth."""
    run_dir = Path(run_dir)
    summary_path = run_dir / 'run_summary.yaml'
    if summary_path.exists():
        summary = yaml.safe_load(summary_path.read_text()) or {}
        for record in summary.get('depth_exports', []):
            requested_depth = float(record.get('requested_depth_m', -1.0))
            if abs(requested_depth - float(depth_m)) < 0.5:
                suffix = str(record.get('suffix', '')).strip()
                label = str(record.get('label', suffix)).strip() or suffix
                if suffix:
                    return suffix, label
    for option_depth, suffix, label in DEPTH_LEVEL_OPTIONS:
        if int(option_depth) == int(depth_m):
            return suffix, label
    raise ValueError(f'Unsupported depth level: {depth_m} m')


def geotiff_leaflet_payload(
    tif_path: Path,
    *,
    value_range_c: tuple[float, float] = TEMPERATURE_RANGE_C,
) -> tuple[str, tuple[tuple[float, float], tuple[float, float]]]:
    """Convert a single-band GeoTIFF into a clipped Leaflet image data URI and bounds."""
    with rasterio.open(tif_path) as src:
        data = src.read(1, masked=True).astype('float32')
        if src.crs is None:
            west, south, east, north = src.bounds
        else:
            west, south, east, north = transform_bounds(src.crs, 'EPSG:4326', *src.bounds, densify_pts=21)

    values = np.ma.filled(data, np.nan)
    valid = np.isfinite(values)
    if not valid.any():
        raise ValueError(f'Prediction raster has no finite values: {tif_path}')
    low, high = map(float, value_range_c)
    clipped = np.clip(values, low, high)
    normalized = np.clip((clipped - low) / (high - low), 0.0, 1.0)

    # Leaflet receives an RGBA PNG because browsers cannot render GeoTIFFs directly.
    rgba = TEMPERATURE_CMAP(normalized)
    rgba[..., 3] = np.where(valid, 0.78, 0.0)
    buffer = BytesIO()
    plt.imsave(buffer, rgba, format='png')
    data_uri = 'data:image/png;base64,' + base64.b64encode(buffer.getvalue()).decode('ascii')
    return data_uri, ((south, west), (north, east))


def temperature_colorbar_payload() -> str:
    """Return a compact color-bar PNG data URI for the fixed temperature scale."""
    fig, ax = plt.subplots(figsize=(5.8, 0.7))
    fig.subplots_adjust(bottom=0.45, top=0.82, left=0.08, right=0.98)
    norm = Normalize(vmin=TEMPERATURE_RANGE_C[0], vmax=TEMPERATURE_RANGE_C[1])
    colorbar = fig.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap=TEMPERATURE_CMAP),
        cax=ax,
        orientation='horizontal',
    )
    colorbar.set_label('Temperature (°C)')
    buffer = BytesIO()
    fig.savefig(buffer, format='png', dpi=140, transparent=True)
    plt.close(fig)
    return 'data:image/png;base64,' + base64.b64encode(buffer.getvalue()).decode('ascii')


def add_geojson_layer(map_obj: Map, path: Path, name: str) -> None:
    """Add a GeoJSON layer to a Leaflet map when the file exists."""
    path = Path(path)
    if path.exists():
        map_obj.add_layer(GeoJSON(data=json.loads(path.read_text()), name=name))


def build_result_map(run_dir: Path, rectangle: tuple[float, float, float, float], *, depth_suffix: str = 'surface', depth_label: str = 'Surface (0 m)') -> Map:
    """Build a Leaflet result map with the prediction overlay and ARGO points."""
    tif_path = prediction_tif_from_run(run_dir, suffix=depth_suffix)
    data_uri, bounds = geotiff_leaflet_payload(tif_path)
    (south, west), (north, east) = bounds
    center = ((south + north) / 2.0, (west + east) / 2.0)
    result_map = Map(center=center, zoom=4, basemap=basemaps.Esri.WorldImagery, scroll_wheel_zoom=True)
    result_map.add_layer(ImageOverlay(url=data_uri, bounds=bounds, name=f'DepthDif {depth_label}'))
    add_geojson_layer(result_map, next(iter(sorted(Path(run_dir).glob('*_argo_points.geojson'))), Path('__missing__')), 'ARGO observations')
    add_geojson_layer(result_map, next(iter(sorted(Path(run_dir).glob('*_patches.geojson'))), Path('__missing__')), 'Selected patches')
    result_map.add_control(LayersControl(position='topright'))
    return result_map


def run_depthdif_inference() -> None:
    """Run public API inference with automatic ARGO and optional OSTIA downloads."""
    _sync_runtime_settings()
    if not STATE.get('skip_ostia', False):
        _sync_copernicus_environment(required=True)
    if STATE.get('rectangle') is None:
        raise ValueError('Please draw a rectangle on the map before running inference.')
    if 'year_widget' not in STATE or 'week_widget' not in STATE:
        raise ValueError('Please run the week-and-map selection step before inference.')
    if STATE.get('assets') is None:
        resolve_depthdif_assets()

    batch_size = int(STATE['batch_size'])
    run_dir = run_week_inference(
        year=int(STATE['year_widget'].value),
        iso_week=int(STATE['week_widget'].value),
        rectangle=STATE['rectangle'],
        output_root=STATE['output_root'],
        device=STATE['device'],
        checkpoint=STATE['assets'].checkpoint,
        config_repo=STATE['hf_repo_id'],
        revision=STATE['hf_revision'],
        cache_dir=STATE['cache_dir'],
        auto_download_argo=True,
        auto_download_ostia=not bool(STATE.get('skip_ostia', False)),
        batch_size=batch_size,
        force_download=False,
    )
    STATE['run_dir'] = Path(run_dir)
    display(_html_status(f'✅ Finished! Your result files are in: {STATE["run_dir"]}', 'ok'))


def display_depthdif_result(depth_m: int = 0) -> None:
    """Display the exported DepthDif prediction on a Leaflet map."""
    if STATE.get('run_dir') is None:
        raise ValueError('Run run_depthdif_inference() first.')
    depth_values = [depth for depth, _suffix, _label in DEPTH_LEVEL_OPTIONS]
    selected_depth = int(depth_m) if int(depth_m) in depth_values else 0
    depth_slider = widgets.SelectionSlider(
        options=[(label, depth) for depth, _suffix, label in DEPTH_LEVEL_OPTIONS],
        value=selected_depth,
        description='Depth',
        continuous_update=False,
        layout=widgets.Layout(width='620px'),
    )
    output = widgets.Output()

    def render_depth(change: dict | None = None) -> None:
        """Refresh the map when the selected depth changes."""
        with output:
            clear_output(wait=True)
            suffix, label = depth_level_metadata(STATE['run_dir'], int(depth_slider.value))
            display(_html_status(f'🌡️ Showing {label}, clipped to 0-30 °C.', 'info'))
            display(HTML(f'<img src="{temperature_colorbar_payload()}" alt="Temperature color scale from 0 to 30 degrees Celsius" style="max-width:620px;width:100%;">'))
            result_map = build_result_map(
                STATE['run_dir'],
                STATE['rectangle'],
                depth_suffix=suffix,
                depth_label=label,
            )
            display(result_map)

    depth_slider.observe(render_depth, names='value')
    render_depth()
    display(widgets.VBox([depth_slider, output]))


## 🛠️ 2. Choose how the run should behave
Pick where temporary files and outputs should live, then choose the compute device and batch size. The default settings are a good starting point: **auto** lets Colab choose the best available device, and batch size **4** is a practical balance for most sessions. Existing matching files are reused automatically, while missing files are downloaded for you. 📦


In [ ]:
configure_depthdif_runtime() # choose folders, device, and batch size
resolve_depthdif_assets() # download the public DepthDif model files

In [ ]:
authenticate_copernicusmarine() # include OSTIA with login, or skip it for ARGO-only

## 🚀 3. Create your ocean map

### 🗺️ 3.1 Pick a week and draw your area
Choose the year and ISO week you want to explore, then draw a rectangle around your ocean region. Smaller areas finish faster and are great for trying things out; larger areas give broader maps but take longer. As a rough guide, a whole-world run can take about an hour, so start with a focused region if you are experimenting. 🌍

In [ ]:
select_week_and_bbox()

### 🚀 3.2 Let DepthDif do the work
Now the notebook gathers the needed ocean observations, runs the model, and writes the map files. If OSTIA was skipped, the model will rely on ARGO observations only; otherwise it will also use the sea-surface temperature layer. 🧪

In [ ]:
run_depthdif_inference() # build the temperature prediction map

## 🧭 4. Explore the result
Open the finished prediction on an interactive map. Use the depth slider to move from the surface down to deeper layers, and read the colors on a fixed 0-30 °C scale: blue is cooler, yellow is mid-range, and red is warmer. You can also zoom, pan, and toggle the ARGO observation points and selected patch layer to see where the model had direct measurements to work from. 🎨

In [ ]:
display_depthdif_result()
